# ViT5-large LoRA — Inference test set và tính metrics, không train lại

Notebook này dùng **adapter LoRA đã train và đã lưu trước đó** để chạy inference trên `test.parquet`.

Mục tiêu:

- Không chạy lại `trainer.train()`.
- Không tokenize train/validation để huấn luyện.
- Load lại `VietAI/vit5-large` gốc + LoRA adapter đã lưu.
- Generate prediction trên test set.
- Tính cùng bộ metrics với notebook fine-tuned: ROUGE, BLEU, BERTScore, Too Short Rate, Length Ratio.
- Lưu prediction/metrics để so sánh với model gốc hoặc notebook khác.

## Cell 1 — Cài thư viện cần thiết

Cell này cài các thư viện phục vụ inference và đánh giá:

- `transformers`: load ViT5-large.
- `peft`: load LoRA adapter đã fine-tune.
- `evaluate`, `rouge-score`, `sacrebleu`, `bert-score`: tính metrics.
- `sentencepiece`, `accelerate`: hỗ trợ tokenizer/model trên Colab.

Nếu Colab đã có sẵn một số thư viện, cell này vẫn chạy an toàn.

In [1]:
%%capture
!pip install -q -U transformers accelerate peft datasets evaluate rouge-score sacrebleu bert-score sentencepiece pyarrow

## Cell 2 — Import thư viện và cấu hình đường dẫn Colab

Cell này khai báo toàn bộ cấu hình inference.

Bạn cần chú ý 3 đường dẫn chính:

- `TEST_PATH`: nơi đặt file `test.parquet`.
- `ADAPTER_DIR`: thư mục chứa LoRA adapter đã train.
- `OUTPUT_ZIP_PATH`: file zip output cũ nếu bạn chỉ upload file nén chứ chưa giải nén.

Notebook sẽ tự giải nén `OUTPUT_ZIP_PATH` nếu chưa thấy `ADAPTER_DIR`.

In [2]:
import os
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import evaluate

from tqdm.auto import tqdm
from IPython.display import display

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, set_seed
from peft import PeftModel

# Tắt Weights & Biases để Colab không hỏi đăng nhập.
os.environ["WANDB_DISABLED"] = "true"

# Tránh warning tokenizer parallelism.
os.environ["TOKENIZERS_PARALLELISM"] = "false"

set_seed(42)

# =========================
# 1. Model và task
# =========================

MODEL_NAME = "VietAI/vit5-large"
TASK_PREFIX = "vietnamese summary: "

# =========================
# 2. Đường dẫn Colab
# =========================

PROJECT_DIR = "/content/NLP"
DATA_DIR = f"{PROJECT_DIR}/data"
OUTPUT_BASE_DIR = f"{PROJECT_DIR}/output"

# File test thật sự. Khi có test, đặt tại đường dẫn này.
TEST_PATH = f"{DATA_DIR}/test.parquet"

# Thư mục adapter đã train từ notebook A100 40GB.
# Sau khi giải nén output zip cũ, thư mục này thường sẽ tồn tại.
ADAPTER_DIR = f"{OUTPUT_BASE_DIR}/vit5-large-lora-a100-40gb-balanced"

# Nếu bạn đã tải output cũ về máy và upload lại lên Colab dưới dạng zip,
# đặt file zip vào một trong các đường dẫn ứng viên dưới đây.
OUTPUT_ZIP_CANDIDATES = [
    f"{OUTPUT_BASE_DIR}/vit5_large_a100_40gb_outputs.zip",
    f"{PROJECT_DIR}/vit5_large_a100_40gb_outputs.zip",
    "/content/vit5_large_a100_40gb_outputs.zip",
    "/content/drive/MyDrive/NLP/vit5_large_a100_40gb_outputs.zip",
]

# Thư mục lưu kết quả test mới.
TEST_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/vit5_large_a100_40gb_test_inference"
TEST_PREDICTION_CSV = f"{TEST_OUTPUT_DIR}/vit5_large_a100_40gb_test_predictions.csv"
TEST_METRICS_JSON = f"{TEST_OUTPUT_DIR}/vit5_large_a100_40gb_test_metrics.json"
TEST_EXAMPLES_CSV = f"{TEST_OUTPUT_DIR}/vit5_large_a100_40gb_test_examples_for_report.csv"
TEST_ZIP_PATH = f"{OUTPUT_BASE_DIR}/vit5_large_a100_40gb_test_inference_outputs.zip"

# =========================
# 3. Cột dữ liệu
# =========================

SOURCE_COL = "article"
TARGET_COL = "summary"

# Nếu test.parquet chưa có summary/reference, notebook vẫn generate prediction,
# nhưng sẽ bỏ qua bước tính ROUGE/BLEU/BERTScore vì không có reference.
ALLOW_TEST_WITHOUT_REFERENCE = True

# =========================
# 4. Cấu hình inference giống bản fine-tuned
# =========================

MAX_SOURCE_LENGTH = 768
MAX_TARGET_LENGTH = 192

FINAL_NUM_BEAMS = 4
FINAL_MIN_LENGTH = 30
FINAL_NO_REPEAT_NGRAM_SIZE = 3

# Batch generate. Nếu T4 bị OOM, giảm xuống 4, 2 hoặc 1.
# Nếu A100 còn dư VRAM, có thể tăng 16 hoặc 32.
TEST_BATCH_SIZE = 8

# None = chạy toàn bộ test set.
# Có thể đặt 50/100 để chạy thử nhanh.
TEST_MAX_SAMPLES = None

# =========================
# 5. Cấu hình metrics
# =========================

RUN_BERTSCORE = True
BERTSCORE_MODEL_TYPE = "bert-base-multilingual-cased"

# "auto": dùng GPU nếu có. Nếu T4 OOM khi tính BERTScore, đổi thành "cpu".
BERTSCORE_DEVICE = "cuda"

# Batch size khi tính BERTScore. Nếu OOM, giảm về 4, 2 hoặc đổi BERTSCORE_DEVICE="cpu".
METRIC_BATCH_SIZE = 8

TOO_SHORT_MIN_TOKENS = 5
TOO_SHORT_REF_RATIO = 0.5

# =========================
# 6. Tạo thư mục output
# =========================

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)
os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)
print("Test path:", TEST_PATH)
print("Adapter directory:", ADAPTER_DIR)
print("Test output directory:", TEST_OUTPUT_DIR)
print("Generation config -> source:", MAX_SOURCE_LENGTH, "| target:", MAX_TARGET_LENGTH, "| beams:", FINAL_NUM_BEAMS, "| batch:", TEST_BATCH_SIZE)
print("Metric config -> BERTScore:", RUN_BERTSCORE, "| device:", BERTSCORE_DEVICE, "| model:", BERTSCORE_MODEL_TYPE)

Project directory: /content/NLP
Data directory: /content/NLP/data
Test path: /content/NLP/data/test.parquet
Adapter directory: /content/NLP/output/vit5-large-lora-a100-40gb-balanced
Test output directory: /content/NLP/output/vit5_large_a100_40gb_test_inference
Generation config -> source: 768 | target: 192 | beams: 4 | batch: 8
Metric config -> BERTScore: True | device: cuda | model: bert-base-multilingual-cased


## Cell 3 — Kiểm tra GPU và khôi phục adapter từ output đã lưu

Cell này kiểm tra GPU hiện tại và kiểm tra xem thư mục LoRA adapter có tồn tại chưa.

Nếu chưa có `ADAPTER_DIR`, notebook sẽ tìm file zip output cũ, ví dụ:

```text
/content/NLP/output/vit5_large_a100_40gb_outputs.zip
```

Nếu tìm thấy, notebook tự giải nén vào `/content/NLP/output`.

Điều này giúp bạn dùng lại trọng số đã train mà **không cần train lại từ đầu**.

In [3]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print("GPU:", gpu_name)
    print("GPU total memory: %.2f GB" % total_mem_gb)
    print("BF16 supported:", torch.cuda.is_bf16_supported())

    # Bật TF32 nếu GPU hỗ trợ để tăng tốc matmul.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception as exc:
        print("Không set được float32 matmul precision:", exc)
else:
    print("CẢNH BÁO: Không có GPU. Inference ViT5-large trên CPU sẽ rất chậm.")

adapter_config_path = Path(ADAPTER_DIR) / "adapter_config.json"
adapter_model_safetensors = Path(ADAPTER_DIR) / "adapter_model.safetensors"

if not adapter_config_path.exists():
    print("Chưa tìm thấy adapter_config.json trong:", ADAPTER_DIR)
    print("Đang tìm output zip cũ để giải nén...")

    found_zip = None
    for zip_candidate in OUTPUT_ZIP_CANDIDATES:
        if Path(zip_candidate).exists():
            found_zip = zip_candidate
            break

    if found_zip is not None:
        print("Tìm thấy zip:", found_zip)
        print("Đang giải nén vào:", OUTPUT_BASE_DIR)
        with zipfile.ZipFile(found_zip, "r") as zf:
            zf.extractall(OUTPUT_BASE_DIR)
        print("Đã giải nén xong.")
    else:
        print("Không tìm thấy zip trong các đường dẫn ứng viên:")
        for p in OUTPUT_ZIP_CANDIDATES:
            print("-", p)

# Kiểm tra lại sau khi giải nén.
if not adapter_config_path.exists():
    raise FileNotFoundError(
        "Không tìm thấy LoRA adapter. Hãy upload/giải nén output đã lưu trước đó. "
        f"Notebook đang cần file: {adapter_config_path}"
    )

if not adapter_model_safetensors.exists():
    print("CẢNH BÁO: Không thấy adapter_model.safetensors. Kiểm tra lại thư mục adapter:")
    print(ADAPTER_DIR)
else:
    print("Đã tìm thấy LoRA adapter:")
    print("-", adapter_config_path)
    print("-", adapter_model_safetensors)

CUDA available: True
GPU: NVIDIA L4
GPU total memory: 22.03 GB
BF16 supported: True
Đã tìm thấy LoRA adapter:
- /content/NLP/output/vit5-large-lora-a100-40gb-balanced/adapter_config.json
- /content/NLP/output/vit5-large-lora-a100-40gb-balanced/adapter_model.safetensors


## Cell 4 — Đọc và làm sạch `test.parquet`

Cell này chỉ đọc test set, không đọc train/validation.

Yêu cầu mặc định:

- Cột đầu vào: `article`
- Cột reference: `summary`

Nếu test set chưa có `summary`, notebook vẫn sinh prediction và lưu CSV, nhưng sẽ bỏ qua metrics cần reference như ROUGE/BLEU/BERTScore.

In [4]:
def clean_text(text):
    """Chuẩn hóa nhẹ văn bản bằng cách ép kiểu string và gộp khoảng trắng."""
    return " ".join(str(text).split())


def load_test_parquet(path, source_col=SOURCE_COL, target_col=TARGET_COL):
    """Đọc test.parquet và làm sạch cột article/summary nếu có."""

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Không tìm thấy test file: {path}\n"
            "Hãy upload test.parquet vào /content/NLP/data/test.parquet"
        )

    df = pd.read_parquet(path)
    print("Test raw shape:", df.shape)
    print("Test columns:", df.columns.tolist())

    if source_col not in df.columns:
        raise ValueError(
            f"Thiếu cột đầu vào {source_col!r}. "
            f"Các cột hiện có: {df.columns.tolist()}"
        )

    has_reference = target_col in df.columns

    if not has_reference:
        if not ALLOW_TEST_WITHOUT_REFERENCE:
            raise ValueError(
                f"Thiếu cột reference {target_col!r}. "
                "Nếu chỉ muốn generate prediction, đặt ALLOW_TEST_WITHOUT_REFERENCE=True."
            )
        print(f"CẢNH BÁO: Không thấy cột {target_col!r}. Notebook sẽ chỉ generate prediction, không tính ROUGE/BLEU/BERTScore.")
        keep_cols = [source_col]
    else:
        keep_cols = [source_col, target_col]

    df = df[keep_cols].dropna(subset=[source_col]).copy()
    df[source_col] = df[source_col].map(clean_text)

    if has_reference:
        df = df.dropna(subset=[target_col]).copy()
        df[target_col] = df[target_col].map(clean_text)
        df = df[(df[source_col].str.len() > 0) & (df[target_col].str.len() > 0)]
    else:
        df = df[df[source_col].str.len() > 0]

    df = df.reset_index(drop=True)

    if TEST_MAX_SAMPLES is not None:
        df = df.head(TEST_MAX_SAMPLES).reset_index(drop=True)

    print("Test used shape:", df.shape)
    print("Has reference:", has_reference)

    return df, has_reference


test_df, has_reference = load_test_parquet(TEST_PATH)
display(test_df.head(3))

Test raw shape: (1344, 2)
Test columns: ['article', 'summary']
Test used shape: (1344, 2)
Has reference: True


,article,summary
0,Văn phòng mới của MayTrip tại địa chỉ 833 Lê H...,MayTrip khai trương văn phòng mới tại TP. HCM ...
1,"Đầu tháng 11, dã quỳ bung nở trên các vạt núi ...",Rừng hoa dã quỳ ở Vườn quốc gia Ba Vì rộng kho...
2,"Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ...","Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ..."


## Cell 5 — Load base model + LoRA adapter đã fine-tune

Cell này load lại model theo đúng cơ chế LoRA:

```text
VietAI/vit5-large gốc + adapter LoRA đã lưu
```

Không có bước train nào trong cell này.

Nếu thư mục adapter có tokenizer đi kèm, notebook ưu tiên load tokenizer từ adapter. Nếu không, tokenizer được load từ `VietAI/vit5-large`.

In [5]:
!pip install -U torchao

In [6]:
# Chọn dtype tự động theo GPU.
if torch.cuda.is_available():
    if torch.cuda.is_bf16_supported():
        MODEL_DTYPE = torch.bfloat16
    else:
        MODEL_DTYPE = torch.float16
else:
    MODEL_DTYPE = torch.float32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Ưu tiên tokenizer đã lưu cùng adapter để đảm bảo inference ổn định.
tokenizer_source = ADAPTER_DIR if (Path(ADAPTER_DIR) / "tokenizer_config.json").exists() else MODEL_NAME

tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_source,
    use_fast=False,
)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)

# Bật cache cho inference để generate nhanh hơn.
model.config.use_cache = True

model.to(device)
model.eval()

print("Tokenizer loaded from:", tokenizer_source)
print("Base model loaded from:", MODEL_NAME)
print("LoRA adapter loaded from:", ADAPTER_DIR)
print("Model dtype:", MODEL_DTYPE)
print("Device:", device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/560 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Tokenizer loaded from: /content/NLP/output/vit5-large-lora-a100-40gb-balanced
Base model loaded from: VietAI/vit5-large
LoRA adapter loaded from: /content/NLP/output/vit5-large-lora-a100-40gb-balanced
Model dtype: torch.bfloat16
Device: cuda


## Cell 6 — Hàm generate summary theo batch

Cell này định nghĩa hàm `generate_summaries`.

Hàm nhận danh sách article, thêm prefix nhiệm vụ `vietnamese summary:`, tokenize, rồi gọi `model.generate`.

Các tham số generate được giữ giống notebook fine-tuned:

- `num_beams = 4`
- `min_length = 30`
- `no_repeat_ngram_size = 3`
- `max_length = 192`

In [7]:
def generate_summaries(texts, batch_size=TEST_BATCH_SIZE):
    """Generate summary cho danh sách article."""

    predictions = []

    for start_idx in tqdm(range(0, len(texts), batch_size), desc="Generating"):
        batch_texts = texts[start_idx:start_idx + batch_size]

        batch_inputs = [
            TASK_PREFIX + clean_text(text)
            for text in batch_texts
        ]

        inputs = tokenizer(
            batch_inputs,
            max_length=MAX_SOURCE_LENGTH,
            truncation=True,
            padding=True,
            return_tensors="pt",
        )

        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.no_grad():
            output_ids = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=MAX_TARGET_LENGTH,
                min_length=FINAL_MIN_LENGTH,
                num_beams=FINAL_NUM_BEAMS,
                length_penalty=1.0,
                no_repeat_ngram_size=FINAL_NO_REPEAT_NGRAM_SIZE,
                early_stopping=True,
            )

        batch_predictions = tokenizer.batch_decode(
            output_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        predictions.extend([pred.strip() for pred in batch_predictions])

    return predictions

## Cell 7 — Chạy inference trên test set và lưu prediction

Cell này chạy prediction trên toàn bộ test set.

Kết quả được lưu vào:

```text
/content/NLP/output/vit5_large_a100_40gb_test_inference/vit5_large_a100_40gb_test_predictions.csv
```

CSV này có các cột:

- `article`
- `reference` nếu test có cột `summary`
- `prediction`

In [8]:
test_articles = test_df[SOURCE_COL].tolist()
test_predictions = generate_summaries(test_articles, batch_size=TEST_BATCH_SIZE)

if has_reference:
    test_references = test_df[TARGET_COL].tolist()
    test_pred_df = pd.DataFrame({
        "article": test_articles,
        "reference": test_references,
        "prediction": test_predictions,
    })
else:
    test_references = None
    test_pred_df = pd.DataFrame({
        "article": test_articles,
        "prediction": test_predictions,
    })

test_pred_df.to_csv(TEST_PREDICTION_CSV, index=False, encoding="utf-8-sig")

print("Saved test predictions to:", TEST_PREDICTION_CSV)
display(test_pred_df.head())

Generating:   0%|          | 0/168 [00:00<?, ?it/s]

Saved test predictions to: /content/NLP/output/vit5_large_a100_40gb_test_inference/vit5_large_a100_40gb_test_predictions.csv


,article,reference,prediction
0,Văn phòng mới của MayTrip tại địa chỉ 833 Lê H...,MayTrip khai trương văn phòng mới tại TP. HCM ...,Văn phòng mới của MayTrip tại TP HCM được thiế...
1,"Đầu tháng 11, dã quỳ bung nở trên các vạt núi ...",Rừng hoa dã quỳ ở Vườn quốc gia Ba Vì rộng kho...,"Ngày 11/11, một rừng hoa dã quỳ ở Ba Vì đã đượ..."
2,"Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ...","Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ...",Bài viết giới thiệu về mùa thu đông tại Tam Cố...
3,"Cầm cốc chocolate đen nóng trên tay, nữ du khá...",Du khách Anh Monisha Rajesh bắt đầu hành trình...,Một nữ du khách Anh Monisha Rajesh đã đi bộ từ...
4,"Cồn Én nằm giữa sông Tiền, thuộc xã Tấn Mỹ, rộ...",Cồn Én là một điểm đến du lịch nằm giữa sông T...,Một khu du lịch sinh thái Cồn Én nằm giữa sông...


## Cell 8 — Định nghĩa các hàm tính metric

Cell này tạo các hàm tính:

- ROUGE-1, ROUGE-2, ROUGE-L, ROUGE-Lsum
- BLEU
- BERTScore Precision/Recall/F1
- Empty Prediction Rate
- Too Short Rate
- Average Prediction Tokens
- Average Reference Tokens
- Average Length Ratio

Các hàm này giống logic metric trong notebook fine-tuned, để kết quả test có thể so sánh công bằng.

In [9]:
from sacrebleu import corpus_bleu
from bert_score import score as bert_score

rouge = evaluate.load("rouge")


def metric_tokenize(text):
    """Tokenize đơn giản theo khoảng trắng để đo độ dài output."""
    return str(text).strip().split()


def compute_rouge_scores(predictions, references):
    """Tính ROUGE cho summarization."""
    if len(predictions) == 0:
        return {
            "rouge1": 0.0,
            "rouge2": 0.0,
            "rougeL": 0.0,
            "rougeLsum": 0.0,
        }

    result = rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=False,
    )

    return {key: float(value) for key, value in result.items()}


def compute_bleu_score(predictions, references):
    """Tính corpus BLEU bằng sacreBLEU."""
    if len(predictions) == 0:
        return {"bleu": 0.0}

    bleu = corpus_bleu(
        predictions,
        [references],
        tokenize="intl",
    )

    return {"bleu": float(bleu.score)}


def get_bertscore_device():
    """Chọn device cho BERTScore."""
    if BERTSCORE_DEVICE == "auto":
        return "cuda" if torch.cuda.is_available() else "cpu"
    return BERTSCORE_DEVICE


def compute_bertscore_scores(predictions, references):
    """Tính BERTScore Precision/Recall/F1 trung bình."""
    if len(predictions) == 0:
        return {
            "bertscore_precision": 0.0,
            "bertscore_recall": 0.0,
            "bertscore_f1": 0.0,
        }

    bert_device = get_bertscore_device()

    precision, recall, f1 = bert_score(
        cands=predictions,
        refs=references,
        model_type=BERTSCORE_MODEL_TYPE,
        device=bert_device,
        batch_size=METRIC_BATCH_SIZE,
        verbose=True,
        rescale_with_baseline=False,
    )

    return {
        "bertscore_precision": float(precision.mean().item()),
        "bertscore_recall": float(recall.mean().item()),
        "bertscore_f1": float(f1.mean().item()),
    }


def compute_too_short_rate(predictions, references=None):
    """Tính các metric về độ dài prediction."""

    if len(predictions) == 0:
        result = {
            "avg_prediction_tokens": 0.0,
            "empty_prediction_rate": 0.0,
            "too_short_rate": 0.0,
            "too_short_min_tokens": TOO_SHORT_MIN_TOKENS,
            "too_short_ref_ratio": TOO_SHORT_REF_RATIO,
        }
        if references is not None:
            result.update({
                "avg_reference_tokens": 0.0,
                "avg_length_ratio_pred_ref": 0.0,
            })
        return result

    pred_lengths = [len(metric_tokenize(pred)) for pred in predictions]
    empty_flags = [length == 0 for length in pred_lengths]

    result = {
        "avg_prediction_tokens": float(np.mean(pred_lengths)),
        "empty_prediction_rate": float(np.mean(empty_flags)),
        "too_short_min_tokens": TOO_SHORT_MIN_TOKENS,
        "too_short_ref_ratio": TOO_SHORT_REF_RATIO,
    }

    if references is None:
        too_short_flags = [pred_len < TOO_SHORT_MIN_TOKENS for pred_len in pred_lengths]
        result["too_short_rate"] = float(np.mean(too_short_flags))
        return result

    ref_lengths = [len(metric_tokenize(ref)) for ref in references]
    too_short_flags = []
    length_ratios = []

    for pred_len, ref_len in zip(pred_lengths, ref_lengths):
        safe_ref_len = max(1, ref_len)
        length_ratio = pred_len / safe_ref_len
        length_ratios.append(length_ratio)

        is_too_short = (
            pred_len < TOO_SHORT_MIN_TOKENS
            or pred_len < TOO_SHORT_REF_RATIO * safe_ref_len
        )
        too_short_flags.append(is_too_short)

    result.update({
        "avg_reference_tokens": float(np.mean(ref_lengths)),
        "avg_length_ratio_pred_ref": float(np.mean(length_ratios)),
        "too_short_rate": float(np.mean(too_short_flags)),
    })

    return result

## Cell 9 — Tính metrics trên test set

Cell này tính metrics nếu test set có cột `summary`.

Nếu test set không có reference, cell vẫn lưu một file JSON tối thiểu gồm số mẫu và các metric độ dài prediction.

In [10]:
test_predictions_list = test_pred_df["prediction"].fillna("").astype(str).tolist()

if has_reference:
    test_references_list = test_pred_df["reference"].fillna("").astype(str).tolist()

    test_rouge_metrics = compute_rouge_scores(
        predictions=test_predictions_list,
        references=test_references_list,
    )

    test_bleu_metrics = compute_bleu_score(
        predictions=test_predictions_list,
        references=test_references_list,
    )

    test_length_metrics = compute_too_short_rate(
        predictions=test_predictions_list,
        references=test_references_list,
    )

    if RUN_BERTSCORE:
        test_bertscore_metrics = compute_bertscore_scores(
            predictions=test_predictions_list,
            references=test_references_list,
        )
    else:
        test_bertscore_metrics = {
            "bertscore_precision": None,
            "bertscore_recall": None,
            "bertscore_f1": None,
        }

    test_metrics = {
        "num_test_samples": len(test_pred_df),
        "model_name": MODEL_NAME,
        "adapter_dir": ADAPTER_DIR,
        "fine_tuning_method": "LoRA",
        "evaluation_split": "test",
        "has_reference": True,
        "max_source_length": MAX_SOURCE_LENGTH,
        "max_target_length": MAX_TARGET_LENGTH,
        "num_beams": FINAL_NUM_BEAMS,
        "min_length": FINAL_MIN_LENGTH,
        "no_repeat_ngram_size": FINAL_NO_REPEAT_NGRAM_SIZE,
        "run_bertscore": RUN_BERTSCORE,
        "bert_score_model": BERTSCORE_MODEL_TYPE if RUN_BERTSCORE else None,
        **test_rouge_metrics,
        **test_bertscore_metrics,
        **test_bleu_metrics,
        **test_length_metrics,
    }
else:
    test_length_metrics = compute_too_short_rate(
        predictions=test_predictions_list,
        references=None,
    )

    test_metrics = {
        "num_test_samples": len(test_pred_df),
        "model_name": MODEL_NAME,
        "adapter_dir": ADAPTER_DIR,
        "fine_tuning_method": "LoRA",
        "evaluation_split": "test",
        "has_reference": False,
        "note": "test.parquet không có cột summary/reference nên không tính ROUGE/BLEU/BERTScore.",
        "max_source_length": MAX_SOURCE_LENGTH,
        "max_target_length": MAX_TARGET_LENGTH,
        "num_beams": FINAL_NUM_BEAMS,
        "min_length": FINAL_MIN_LENGTH,
        "no_repeat_ngram_size": FINAL_NO_REPEAT_NGRAM_SIZE,
        **test_length_metrics,
    }

print("===== TEST METRICS =====")
print(json.dumps(test_metrics, ensure_ascii=False, indent=2))

with open(TEST_METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, ensure_ascii=False, indent=2)

print("Saved test metrics to:", TEST_METRICS_JSON)

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/336 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/168 [00:00<?, ?it/s]

done in 6.52 seconds, 206.25 sentences/sec
===== TEST METRICS =====
{
  "num_test_samples": 1344,
  "model_name": "VietAI/vit5-large",
  "adapter_dir": "/content/NLP/output/vit5-large-lora-a100-40gb-balanced",
  "fine_tuning_method": "LoRA",
  "evaluation_split": "test",
  "has_reference": true,
  "max_source_length": 768,
  "max_target_length": 192,
  "num_beams": 4,
  "min_length": 30,
  "no_repeat_ngram_size": 3,
  "run_bertscore": true,
  "bert_score_model": "bert-base-multilingual-cased",
  "rouge1": 0.683470746066039,
  "rouge2": 0.404861688948587,
  "rougeL": 0.4396512322140551,
  "rougeLsum": 0.4396781473289817,
  "bertscore_precision": 0.7782228589057922,
  "bertscore_recall": 0.7635817527770996,
  "bertscore_f1": 0.7705584764480591,
  "bleu": 22.12972564356505,
  "avg_prediction_tokens": 79.84449404761905,
  "empty_prediction_rate": 0.0,
  "too_short_min_tokens": 5,
  "too_short_ref_ratio": 0.5,
  "avg_reference_tokens": 103.3139880952381,
  "avg_length_ratio_pred_ref": 0.805

## Cell 10 — Lưu một số ví dụ định tính cho báo cáo

Cell này lấy vài dòng đầu của test prediction để bạn đọc nhanh và đưa vào báo cáo.

Nếu có reference, bảng sẽ gồm:

```text
article | reference | prediction
```

Nếu không có reference, bảng chỉ gồm:

```text
article | prediction
```

In [11]:
num_examples = min(20, len(test_pred_df))

test_examples_df = test_pred_df.head(num_examples).copy()
test_examples_df.to_csv(TEST_EXAMPLES_CSV, index=False, encoding="utf-8-sig")

print("Saved examples to:", TEST_EXAMPLES_CSV)
display(test_examples_df)

Saved examples to: /content/NLP/output/vit5_large_a100_40gb_test_inference/vit5_large_a100_40gb_test_examples_for_report.csv


,article,reference,prediction
0,Văn phòng mới của MayTrip tại địa chỉ 833 Lê H...,MayTrip khai trương văn phòng mới tại TP. HCM ...,Văn phòng mới của MayTrip tại TP HCM được thiế...
1,"Đầu tháng 11, dã quỳ bung nở trên các vạt núi ...",Rừng hoa dã quỳ ở Vườn quốc gia Ba Vì rộng kho...,"Ngày 11/11, một rừng hoa dã quỳ ở Ba Vì đã đượ..."
2,"Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ...","Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ...",Bài viết giới thiệu về mùa thu đông tại Tam Cố...
3,"Cầm cốc chocolate đen nóng trên tay, nữ du khá...",Du khách Anh Monisha Rajesh bắt đầu hành trình...,Một nữ du khách Anh Monisha Rajesh đã đi bộ từ...
4,"Cồn Én nằm giữa sông Tiền, thuộc xã Tấn Mỹ, rộ...",Cồn Én là một điểm đến du lịch nằm giữa sông T...,Một khu du lịch sinh thái Cồn Én nằm giữa sông...
5,"Buổi biểu diễn do Sun Group đầu tư, kéo dài gầ...",Một buổi biểu diễn đặc biệt được tổ chức tại T...,"Ngày 19/4/2024, buổi biểu diễn do Sun Group và..."
6,The Imperial Hotel - tọa lạc trung tâm Vũng Tà...,Cầu vượt Imperial tại The Imperial Hotel ở Vũn...,"Ngày nay, The Imperial Hotel là một trong nhữn..."
7,Điểm trekking được nhiều người trẻ ưa thích cò...,"Núi Dinh, còn được gọi là núi Ông Trịnh, là mộ...",Một trong những điểm trekking được nhiều người...
8,Hai giải thưởng gồm 'Khu nghỉ dưỡng có thiết k...,Amiana Resort Nha Trang đã giành hai giải thưở...,"Ngày nay, Amiana Resort Nha Trang là một khu n..."
9,Lễ ký kết hợp tác diễn ra tại thủ đô Kuala Lum...,Lễ ký kết hợp tác giữa Việt Nam và Malaysia di...,Lễ ký kết hợp tác du lịch giữa Việt Nam và Mal...


## Cell 11 — So sánh nhanh với một file metrics khác nếu có

Cell này là tùy chọn.

Nếu bạn đã có metrics test của model gốc hoặc model khác, hãy sửa `OTHER_METRICS_JSON` trỏ tới file đó. Notebook sẽ tạo bảng so sánh:

```text
other_model
current_lora_model
current_minus_other
```

Nếu chưa có file khác, cell sẽ bỏ qua.

In [12]:
# Sửa path này nếu bạn muốn so sánh với metrics của model gốc hoặc notebook khác.
OTHER_METRICS_JSON = None
# Ví dụ:
# OTHER_METRICS_JSON = "/content/NLP/output/vit5_large_origin_test_metrics.json"

if OTHER_METRICS_JSON is None or not Path(OTHER_METRICS_JSON).exists():
    print("Chưa có OTHER_METRICS_JSON để so sánh. Bỏ qua cell so sánh.")
else:
    with open(OTHER_METRICS_JSON, "r", encoding="utf-8") as f:
        other_metrics = json.load(f)

    with open(TEST_METRICS_JSON, "r", encoding="utf-8") as f:
        current_metrics = json.load(f)

    metric_keys = [
        "rouge1",
        "rouge2",
        "rougeL",
        "rougeLsum",
        "bleu",
        "bertscore_precision",
        "bertscore_recall",
        "bertscore_f1",
        "too_short_rate",
        "empty_prediction_rate",
        "avg_prediction_tokens",
        "avg_reference_tokens",
        "avg_length_ratio_pred_ref",
    ]

    rows = []
    for key in metric_keys:
        if key in other_metrics or key in current_metrics:
            other_value = other_metrics.get(key)
            current_value = current_metrics.get(key)

            if isinstance(other_value, (int, float)) and isinstance(current_value, (int, float)):
                diff = current_value - other_value
            else:
                diff = None

            rows.append({
                "metric": key,
                "other_model": other_value,
                "current_lora_model": current_value,
                "current_minus_other": diff,
            })

    compare_df = pd.DataFrame(rows)
    compare_csv = f"{TEST_OUTPUT_DIR}/test_metric_comparison.csv"
    compare_df.to_csv(compare_csv, index=False, encoding="utf-8-sig")

    print("Saved comparison to:", compare_csv)
    display(compare_df)

Chưa có OTHER_METRICS_JSON để so sánh. Bỏ qua cell so sánh.


## Cell 12 — Nén kết quả test để tải về

Cell này nén các file test inference:

- Prediction CSV
- Metrics JSON
- Examples CSV
- Comparison CSV nếu có

File zip được lưu tại:

```text
/content/NLP/output/vit5_large_a100_40gb_test_inference_outputs.zip
```

Bạn nên tải file zip này về máy hoặc copy sang Google Drive để tránh mất output khi Colab tắt runtime.

In [13]:
items_to_zip = [
    Path(TEST_PREDICTION_CSV),
    Path(TEST_METRICS_JSON),
    Path(TEST_EXAMPLES_CSV),
    Path(f"{TEST_OUTPUT_DIR}/test_metric_comparison.csv"),
]

zip_path = Path(TEST_ZIP_PATH)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for item in items_to_zip:
        if item.exists() and item.is_file():
            zf.write(
                item,
                arcname=item.relative_to(OUTPUT_BASE_DIR),
            )

print("Output zip:", zip_path)

Output zip: /content/NLP/output/vit5_large_a100_40gb_test_inference_outputs.zip
